In [1]:
import pandas as pd

In [6]:
cs = pd.read_csv(
    "../data/true_sources_final.csv",
    dtype={"source_id": "string"},
)

cs["source_id"] = cs["source_id"].str.strip()

# Keep a record of duplicate input rows for inspection
duplicate_rows = cs[
    cs.duplicated("source_id", keep=False)
].sort_values("source_id")

duplicate_rows.to_csv("../data/duplicate_source_rows.csv", index=False)

# Temporary ranking columns
cs["_syd_success_num"] = pd.to_numeric(
    cs["syd_success"],
    errors="coerce",
)

cs["_numax_err_num"] = pd.to_numeric(
    cs["numax_sig"],
    errors="coerce",
)

cs["_preferred_syd"] = cs["_syd_success_num"].eq(2)

# Sort so the preferred row is first for each source_id
cs = (
    cs.sort_values(
        by=["source_id", "_preferred_syd", "_numax_err_num"],
        ascending=[True, False, True],
        na_position="last",
        kind="stable",
    )
    .drop_duplicates("source_id", keep="first")
    .drop(columns=[
        "_syd_success_num",
        "_numax_err_num",
        "_preferred_syd",
    ])
    .reset_index(drop=True)
)

print(f"Duplicate rows removed: {len(duplicate_rows) - len(cs[cs['source_id'].isin(duplicate_rows['source_id'])])}")
print(f"Remaining stars: {len(cs)}")

Duplicate rows removed: 10
Remaining stars: 2581


In [ ]:
#!/usr/bin/env python3

from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd


# ============================================================
# Paths
# ============================================================

BASE_DIR = Path("/Users/z5214708/BASTA/BASTA/final_runs")

# This must be the duplicate-bearing file used to create the ASCII files.
MASTER_FILE = "../data/true_sources_w_duplicates.csv"

INPUT_DIR = BASE_DIR / "data" 
OUTPUT_ROOT = BASE_DIR / "output"


# Existing BASTA input files
INPUT_FILE_TEMPLATES = {
    1: "raw_{letter}.ascii",
    2: "raw_dnu_{letter}.ascii",
    3: "corrected_{letter}.ascii",
    4: "corrected_dnu_{letter}.ascii",
}


# Existing BASTA output files to clean
BASTA_OUTPUT_FILES = (
    "results.ascii",
    "results_distance.ascii",
)


# If an exact row match cannot be established, keep the first
# output row but record it in the selection log.
KEEP_FIRST_IF_UNRESOLVED = True


# ============================================================
# Column definitions
# ============================================================

INPUT_COLUMNS = [
    "starid",
    "numax",
    "numax_err",
    "dnu",
    "dnu_err",
    "Teff",
    "Teff_err",
    "FeH",
    "FeH_err",
    "RA",
    "DEC",
    "G_GAIA",
    "G_GAIA_err",
    "Mj_2MASS",
    "Mj_2MASS_err",
    "Mh_2MASS",
    "Mh_2MASS_err",
    "Mk_2MASS",
    "Mk_2MASS_err",
    "phase",
]

BAD_VALUES = [
    "-999.999",
    "-999",
    "-99.999",
    "nan",
    "NaN",
]


# ============================================================
# Utility functions
# ============================================================

def normalise_id(value):
    """Convert source IDs consistently to strings."""
    if value is None or pd.isna(value):
        return None

    value = str(value).strip()
    value = value.strip("'\"")

    # Convert IDs such as 123456.0 to 123456
    if value.endswith(".0") and value[:-2].isdigit():
        value = value[:-2]

    return value


def normalise_id_series(series):
    return series.map(normalise_id)


def rounded_number(value, decimals=8):
    """Create a stable numerical matching value."""
    value = pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]

    if pd.isna(value) or not np.isfinite(value):
        return None

    return round(float(value), decimals)


def read_input_ascii(path):
    """Read one BASTA input ASCII file."""
    data = pd.read_csv(
        path,
        sep=r"\s+",
        names=INPUT_COLUMNS,
        header=None,
        dtype={"starid": "string"},
        na_values=BAD_VALUES,
        engine="python",
    )

    data["_input_row"] = np.arange(len(data))
    data["_id"] = normalise_id_series(data["starid"])

    return data


def read_basta_output(path):
    """
    Read a BASTA output file.

    BASTA output headers often begin with '#', so the header is
    detected manually.
    """
    with path.open("r") as file:
        lines = file.readlines()

    header_index = None
    column_names = None

    for index, line in enumerate(lines):
        stripped = line.lstrip("#").strip()

        if not stripped:
            continue

        tokens = stripped.split()
        lower_tokens = [token.lower() for token in tokens]

        if "starid" in lower_tokens:
            header_index = index
            column_names = tokens
            break

    if header_index is None:
        raise ValueError(
            f"Could not find a starid header in BASTA output:\n{path}"
        )

    output = pd.read_csv(
        path,
        sep=r"\s+",
        skiprows=header_index + 1,
        names=column_names,
        header=None,
        dtype={"starid": "string"},
        comment="#",
        na_values=BAD_VALUES,
        engine="python",
    )

    starid_columns = [
        column for column in output.columns
        if column.lower() == "starid"
    ]

    if not starid_columns:
        raise ValueError(f"No starid column found in {path}")

    if starid_columns[0] != "starid":
        output = output.rename(
            columns={starid_columns[0]: "starid"}
        )

    output["_id"] = normalise_id_series(output["starid"])
    output["_output_row"] = np.arange(len(output))
    output["_output_occurrence"] = (
        output.groupby("_id", sort=False).cumcount()
    )

    return output


# ============================================================
# Master-source preparation
# ============================================================

def prepare_master(source, corrected):
    """
    Prepare the master table and lookup dictionaries.

    For corrected runs, the corrected numax and numax uncertainty
    are calculated exactly as in the user's data-preparation code.
    """
    master = source.copy().reset_index(drop=True)

    if "source_id" not in master.columns:
        raise ValueError("true_sources_final.csv needs a source_id column")

    if "syd_success" not in master.columns:
        raise ValueError(
            "true_sources_final.csv needs a syd_success column"
        )

    if "numax_sig" in master.columns:
        source_numax_error_column = "numax_sig"
    elif "numax_err" in master.columns:
        source_numax_error_column = "numax_err"
    else:
        raise ValueError(
            "Could not find numax_sig or numax_err in master file"
        )

    if "dnu_sig" in master.columns:
        source_dnu_error_column = "dnu_sig"
    elif "dnu_err" in master.columns:
        source_dnu_error_column = "dnu_err"
    else:
        raise ValueError(
            "Could not find dnu_sig or dnu_err in master file"
        )

    master["_id"] = normalise_id_series(master["source_id"])
    master["_source_row"] = np.arange(len(master))

    master["_syd_success_num"] = pd.to_numeric(
        master["syd_success"],
        errors="coerce",
    )

    master["_is_syd2"] = master["_syd_success_num"].eq(2)

    raw_numax = pd.to_numeric(
        master["numax"],
        errors="coerce",
    )

    raw_numax_error = pd.to_numeric(
        master[source_numax_error_column],
        errors="coerce",
    )

    if corrected:
        feh = pd.to_numeric(master["feh"], errors="coerce")
        sfeh = pd.to_numeric(master["s_feh"], errors="coerce")

        factor = -0.0588 * feh + 0.949

        master["_match_numax"] = factor * raw_numax

        master["_match_numax_err"] = np.sqrt(
            (factor * raw_numax_error) ** 2
            + (0.0588 * raw_numax * sfeh) ** 2
        )
    else:
        master["_match_numax"] = raw_numax
        master["_match_numax_err"] = raw_numax_error

    master["_match_dnu"] = pd.to_numeric(
        master["dnu"],
        errors="coerce",
    )

    master["_match_dnu_err"] = pd.to_numeric(
        master[source_dnu_error_column],
        errors="coerce",
    )

    lookup = defaultdict(list)
    by_id = defaultdict(list)

    for index, row in master.iterrows():

        key = (
            row["_id"],
            rounded_number(row["_match_numax"]),
            rounded_number(row["_match_numax_err"]),
            rounded_number(row["_match_dnu"]),
            rounded_number(row["_match_dnu_err"]),
        )

        if row["_id"] is not None:
            lookup[key].append(index)
            by_id[row["_id"]].append(index)

    return master, lookup, by_id


def master_rank(master, index):
    """
    Ranking priority:

    1. syd_success == 2
    2. smaller numax uncertainty
    3. original row order
    """
    is_syd2 = bool(master.at[index, "_is_syd2"])
    error = master.at[index, "_match_numax_err"]

    if pd.isna(error):
        error_missing = 1
        error_value = np.inf
    else:
        error_missing = 0
        error_value = float(error)

    return (
        0 if is_syd2 else 1,
        error_missing,
        error_value,
        int(master.at[index, "_source_row"]),
    )


def relative_distance(input_row, master_row):
    """
    Used only as a fallback if the exact numerical key does not match.
    """
    pairs = [
        ("numax", "_match_numax"),
        ("numax_err", "_match_numax_err"),
        ("dnu", "_match_dnu"),
        ("dnu_err", "_match_dnu_err"),
    ]

    differences = []

    for input_column, master_column in pairs:
        input_value = pd.to_numeric(
            pd.Series([input_row[input_column]]),
            errors="coerce",
        ).iloc[0]

        master_value = pd.to_numeric(
            pd.Series([master_row[master_column]]),
            errors="coerce",
        ).iloc[0]

        if pd.isna(input_value) or pd.isna(master_value):
            continue

        scale = max(abs(float(master_value)), 1.0)

        differences.append(
            abs(float(input_value) - float(master_value)) / scale
        )

    if not differences:
        return np.inf

    return float(np.mean(differences))


# ============================================================
# Match input rows to the master file
# ============================================================

def annotate_input(input_data, master, lookup, by_id):
    """
    Attach syd_success and the appropriate numax uncertainty to
    every input ASCII row.
    """
    matched_rows = []
    match_statuses = []

    for _, input_row in input_data.iterrows():

        key = (
            input_row["_id"],
            rounded_number(input_row["numax"]),
            rounded_number(input_row["numax_err"]),
            rounded_number(input_row["dnu"]),
            rounded_number(input_row["dnu_err"]),
        )

        candidates = lookup.get(key, [])

        if candidates:
            selected = min(
                candidates,
                key=lambda index: master_rank(master, index),
            )

            matched_rows.append(selected)
            match_statuses.append("exact")

            continue

        # Fallback: use the same source ID and nearest numerical values
        candidates = by_id.get(input_row["_id"], [])

        if candidates:
            selected = min(
                candidates,
                key=lambda index: (
                    relative_distance(
                        input_row,
                        master.loc[index],
                    ),
                    master_rank(master, index),
                ),
            )

            matched_rows.append(selected)
            match_statuses.append("source_id_fallback")

        else:
            matched_rows.append(None)
            match_statuses.append("unmatched")

    input_data = input_data.copy()

    input_data["_master_row"] = matched_rows
    input_data["_match_status"] = match_statuses

    input_data["_syd_success"] = np.nan
    input_data["_run_numax_err"] = np.nan

    for input_index, master_index in enumerate(matched_rows):

        if master_index is None:
            continue

        input_data.loc[
            input_data.index[input_index],
            "_syd_success",
        ] = master.loc[master_index, "_syd_success_num"]

        input_data.loc[
            input_data.index[input_index],
            "_run_numax_err",
        ] = master.loc[master_index, "_match_numax_err"]

    return input_data


def choose_preferred_input_rows(input_data):
    """
    Choose one preferred input row per star.

    Preference:
    1. syd_success == 2
    2. smaller numax uncertainty
    3. original input row order
    """
    input_data = input_data.copy()

    input_data["_input_occurrence"] = (
        input_data.groupby("_id", sort=False).cumcount()
    )

    input_data["_is_syd2"] = (
        pd.to_numeric(
            input_data["_syd_success"],
            errors="coerce",
        ).eq(2)
    )

    input_data["_error_missing"] = (
        input_data["_run_numax_err"].isna()
    )

    ordered = input_data.sort_values(
        by=[
            "_id",
            "_is_syd2",
            "_error_missing",
            "_run_numax_err",
            "_input_row",
        ],
        ascending=[
            True,
            False,
            True,
            True,
            True,
        ],
        kind="stable",
    )

    preferred = ordered.drop_duplicates(
        subset="_id",
        keep="first",
    )

    preferred_occurrence = dict(
        zip(
            preferred["_id"],
            preferred["_input_occurrence"],
        )
    )
    return input_data, preferred, preferred_occurrence


# ============================================================
# Clean BASTA output
# ============================================================

def clean_basta_output(
    output_path,
    input_data,
    preferred_occurrence,
):
    """
    Select one BASTA output row per star.

    If the number of output rows equals the number of input rows
    for a duplicated star, occurrence order is used to identify the
    correct output row.
    """
    output = read_basta_output(output_path)

    input_counts = (
        input_data.groupby("_id", sort=False)
        .size()
        .to_dict()
    )

    selected_output_rows = []
    selection_log = []

    groups = output.groupby(
        "_id",
        sort=False,
        dropna=False,
    )

    for star_id, group in groups:

        n_output = len(group)
        n_input = int(input_counts.get(star_id, 0))

        preferred_input_occurrence = preferred_occurrence.get(
            star_id,
            None,
        )

        if n_output == 1:

            selected_position = 0
            method = "unique_output"
            unresolved = False

        elif (
            preferred_input_occurrence is not None
            and n_input == n_output
            and int(preferred_input_occurrence) < n_output
        ):

            selected_position = int(preferred_input_occurrence)
            method = "input_output_occurrence"
            unresolved = False

        else:

            if not KEEP_FIRST_IF_UNRESOLVED:
                raise RuntimeError(
                    f"Could not safely resolve duplicate output rows "
                    f"for starid={star_id} in {output_path}"
                )

            selected_position = 0
            method = "fallback_first_unresolved"
            unresolved = True

        selected_row = group.iloc[selected_position]

        selected_output_rows.append(
            int(selected_row["_output_row"])
        )

        selection_log.append(
            {
                "starid": selected_row["starid"],
                "n_input_rows": n_input,
                "n_output_rows": n_output,
                "preferred_input_occurrence": (
                    preferred_input_occurrence
                ),
                "selected_output_occurrence": selected_position,
                "method": method,
                "unresolved": unresolved,
            }
        )

    selected_output_rows = sorted(selected_output_rows)

    clean_output = output.loc[
        output["_output_row"].isin(selected_output_rows)
    ].copy()

    clean_output = clean_output.sort_values(
        "_output_row"
    )

    private_columns = [
        column
        for column in clean_output.columns
        if column.startswith("_")
    ]

    clean_output = clean_output.drop(
        columns=private_columns
    )

    clean_path = output_path.with_name(
        output_path.stem + "_clean" + output_path.suffix
    )

    clean_output.to_csv(
        clean_path,
        sep=" ",
        index=False,
        na_rep="-999.999",
    )

    log_path = output_path.with_name(
        output_path.stem + "_selection_log.csv"
    )

    pd.DataFrame(selection_log).to_csv(
        log_path,
        index=False,
    )

    return (
        clean_path,
        log_path,
        pd.DataFrame(selection_log),
    )


In [20]:
source = pd.read_csv(
        MASTER_FILE,
        dtype={"source_id": "string"},
        low_memory=False,
    )

source_ids = (
    source["source_id"]
    .map(normalise_id)
    .dropna()
)

input_data = read_input_ascii(
    INPUT_DIR / "raw_a.ascii"
)

input_ids = (
    input_data["starid"]
    .map(normalise_id)
)

source_id_set = set(source_ids)

print("Example master IDs:")
print(source["source_id"].head().tolist())

print("\nExample input IDs:")
print(input_data["starid"].head().tolist())

print(
    "\nMatching input IDs:",
    input_ids.isin(source_id_set).sum(),
    "of",
    len(input_ids),
)

Example master IDs:
['2051287036390801024', '2051293702180102144', '2051294801692061952', '2051366682266279424', '2051664581201475840']

Example input IDs:
['2051293702180102144', '2051366682266279424', '2051798167564062592', '2051829336143170816', '2052067002458806528']

Matching input IDs: 1847 of 1847


In [21]:


# ============================================================
# Main processing loop
# ============================================================

def main():

    source = pd.read_csv(
        MASTER_FILE,
        dtype={"source_id": "string"},
        low_memory=False,
    )

    source_ids = normalise_id_series(
        source["source_id"]
    )

    duplicate_rows = int(
        source_ids.duplicated(keep=False).sum()
    )

    duplicate_extra_rows = int(
        source_ids.duplicated().sum()
    )

    print(f"Master rows: {len(source)}")
    print(
        "Rows belonging to duplicate source_id groups: "
        f"{duplicate_rows}"
    )
    print(
        "Duplicate rows after the first occurrence: "
        f"{duplicate_extra_rows}"
    )

    if duplicate_rows == 0:
        print(
            "\nWARNING: true_sources_final.csv contains no "
            "duplicate source_id values."
        )
        print(
            "If duplicates were created by a later merge, use the "
            "duplicate-bearing cs table instead."
        )

    master_info = {
        False: prepare_master(
            source,
            corrected=False,
        ),
        True: prepare_master(
            source,
            corrected=True,
        ),
    }

    for table_number in [1, 2, 3, 4]:

        corrected = table_number in [3, 4]

        master, lookup, by_id = master_info[corrected]

        input_template = INPUT_FILE_TEMPLATES[table_number]

        for letter in ["a", "b", "c", "d"]:

            input_path = INPUT_DIR / input_template.format(
                letter=letter
            )

            run_directory = OUTPUT_ROOT / (
                f"table{table_number}_{letter}"
            )

            if not input_path.exists():
                print(f"\nSkipping missing input: {input_path}")
                continue

            if not run_directory.exists():
                print(
                    f"\nSkipping missing output directory: "
                    f"{run_directory}"
                )
                continue

            print(
                f"\nProcessing table {table_number}, "
                f"subset {letter}"
            )

            input_data = read_input_ascii(input_path)

            input_data = annotate_input(
                input_data,
                master,
                lookup,
                by_id,
            )

            (
                input_data,
                preferred_input,
                preferred_occurrence,
            ) = choose_preferred_input_rows(input_data)

            n_input_duplicates = int(
                input_data["_id"].duplicated(keep=False).sum()
            )

            n_unmatched = int(
                (input_data["_match_status"] == "unmatched").sum()
            )

            print(
                f"Input rows: {len(input_data)}; "
                f"duplicate input rows: {n_input_duplicates}; "
                f"unmatched master rows: {n_unmatched}"
            )

            # Save the selected input rows for auditing
            preferred_columns = [
                column
                for column in INPUT_COLUMNS
                if column in preferred_input.columns
            ]

            preferred_columns += [
                "_input_row",
                "_input_occurrence",
                "_syd_success",
                "_run_numax_err",
                "_match_status",
            ]

            preferred_input_path = run_directory / (
                f"preferred_input_{input_path.stem}.csv"
            )

            preferred_input[preferred_columns].sort_values(
                "_input_row"
            ).to_csv(
                preferred_input_path,
                index=False,
            )

            for basta_output_name in BASTA_OUTPUT_FILES:

                output_path = run_directory / basta_output_name

                if not output_path.exists():
                    continue

                try:
                    (
                        clean_path,
                        log_path,
                        selection_log,
                    ) = clean_basta_output(
                        output_path,
                        input_data,
                        preferred_occurrence,
                    )

                    unresolved = int(
                        selection_log["unresolved"].sum()
                    )

                    print(
                        f"  Wrote: {clean_path.name}; "
                        f"unresolved duplicate groups: {unresolved}"
                    )

                except Exception as error:
                    print(
                        f"  ERROR processing {output_path}: "
                        f"{error}"
                    )


if __name__ == "__main__":
    main()

Master rows: 2591
Rows belonging to duplicate source_id groups: 20
Duplicate rows after the first occurrence: 10

Processing table 1, subset a
Input rows: 1847; duplicate input rows: 12; unmatched master rows: 0
  Wrote: results_clean.ascii; unresolved duplicate groups: 0
  Wrote: results_distance_clean.ascii; unresolved duplicate groups: 0

Processing table 1, subset b
Input rows: 628; duplicate input rows: 4; unmatched master rows: 0
  Wrote: results_clean.ascii; unresolved duplicate groups: 0
  Wrote: results_distance_clean.ascii; unresolved duplicate groups: 0

Processing table 1, subset c
Input rows: 83; duplicate input rows: 2; unmatched master rows: 0
  Wrote: results_clean.ascii; unresolved duplicate groups: 0
  Wrote: results_distance_clean.ascii; unresolved duplicate groups: 0

Processing table 1, subset d
Input rows: 33; duplicate input rows: 0; unmatched master rows: 0
  Wrote: results_clean.ascii; unresolved duplicate groups: 0
  Wrote: results_distance_clean.ascii; unreso

In [7]:
ps = pd.read_csv(output_dir/"postprocessing_summary.csv")

In [ ]:
cs = pd.read_csv(
    "../data/true_sources_final.csv",
    dtype={"source_id": "string"},
)

cs.to_csv('../data/true_sources_w_duplicates.csv',index=False)

cs["source_id"] = cs["source_id"].str.strip()

# Keep a record of duplicate input rows for inspection
duplicate_rows = cs[
    cs.duplicated("source_id", keep=False)
].sort_values("source_id")

duplicate_rows.to_csv("../data/duplicate_source_rows.csv", index=False)

# Temporary ranking columns
cs["_syd_success_num"] = pd.to_numeric(
    cs["syd_success"],
    errors="coerce",
)

cs["_numax_err_num"] = pd.to_numeric(
    cs["numax_sig"],
    errors="coerce",
)

cs["_preferred_syd"] = cs["_syd_success_num"].eq(2)

# Sort so the preferred row is first for each source_id
cs = (
    cs.sort_values(
        by=["source_id", "_preferred_syd", "_numax_err_num"],
        ascending=[True, False, True],
        na_position="last",
        kind="stable",
    )
    .drop_duplicates("source_id", keep="first")
    .drop(columns=[
        "_syd_success_num",
        "_numax_err_num",
        "_preferred_syd",
    ])
    .reset_index(drop=True)
)

print(f"Duplicate rows removed: {len(duplicate_rows) - len(cs[cs['source_id'].isin(duplicate_rows['source_id'])])}")
print(f"Remaining stars: {len(cs)}")

Duplicate rows removed: 10
Remaining stars: 2581
